# Clasificador de Calzados Manaco - Entrenamiento Convolucional (CNN)

Este cuaderno entrena el modelo de Visión Artificial para clasificar **3 categorías de calzado de Manaco** más la clase de fondo:
1. `sin_calzado`: Fondo vacío, manos, objetos ajenos
2. `calzado_deportivo`: Zapatillas y calzado deportivo (SA6: 640411)
3. `calzado_casual`: Calzado de cuero con suela sintética/mocasines (SA6: 640399)
4. `calzado_formal`: Calzado formal de vestir con suela y capellada de cuero (SA6: 640420)

> **Nota:** Asegúrate de seleccionar el Kernel de Python del entorno virtual (`.venv`) en la esquina superior derecha.

In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from config import (ARCHIVO_CLASES, ARCHIVO_MODELO, CARPETA_DATASET, CLASES,
                    TAMANO_IMG, preprocesar)

print("TensorFlow version:", tf.__version__)
print("Dispositivos disponibles:", tf.config.list_physical_devices())
print("Clases configuradas:", CLASES)

In [ ]:
# Verificar y cargar dataset desde dataset/
X = []
y = []

for idx, c in enumerate(CLASES):
    carpeta = os.path.join(CARPETA_DATASET, c)
    os.makedirs(carpeta, exist_ok=True)
    archivos = [f for f in os.listdir(carpeta) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Categoría '{c}': {len(archivos)} imágenes")
    for arc in archivos:
        img = cv2.imread(os.path.join(carpeta, arc))
        if img is not None:
            X.append(preprocesar(img))
            y.append(idx)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print(f"\nTotal muestras: {len(X)} con tensor de entrada {X.shape}")

In [ ]:
# Visualizar algunas muestras aleatorias del dataset
if len(X) > 0:
    fig, axes = plt.subplots(1, min(5, len(X)), figsize=(15, 3))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    indices = np.random.choice(len(X), len(axes), replace=False)
    for ax, idx in zip(axes, indices):
        ax.imshow(X[idx])
        ax.set_title(CLASES[y[idx]])
        ax.axis('off')
    plt.show()

In [ ]:
# Partición en entrenamiento y validación (85% / 15%)
perm = np.random.default_rng(42).permutation(len(X))
X = X[perm]
y = y[perm]

corte = max(1, int(len(X) * 0.85))
X_train, X_val = X[:corte], X[corte:]
y_train, y_val = y[:corte], y[corte:]
if len(X_val) == 0:
    X_val, y_val = X_train, y_train

print(f"Entrenamiento: {len(X_train)} | Validación: {len(X_val)}")

In [ ]:
# Definición de la Red Neuronal Convolucional con Data Augmentation
aumento = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomTranslation(0.15, 0.15),
    tf.keras.layers.RandomZoom((-0.2, 0.2)),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 1)),
    tf.keras.layers.RandomContrast(0.2)
], name="data_augmentation")

modelo = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(TAMANO_IMG, TAMANO_IMG, 3)),
    aumento,
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(CLASES), activation='softmax')
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
modelo.summary()

In [ ]:
# Entrenamiento
early_stop = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
historia = modelo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=min(32, len(X_train)),
    epochs=40,
    callbacks=[early_stop]
)

In [ ]:
# Curvas de aprendizaje y Matriz de Confusión
pred_probs = modelo.predict(X_val)
pred_y = np.argmax(pred_probs, axis=1)
matriz = tf.math.confusion_matrix(y_val, pred_y, num_classes=len(CLASES)).numpy()

print("Matriz de Confusión:")
print(CLASES)
print(matriz)

# Guardar modelo
modelo.save(ARCHIVO_MODELO)
with open(ARCHIVO_CLASES, 'w', encoding='utf-8') as f:
    json.dump(CLASES, f, indent=2, ensure_ascii=False)
print(f"\n[OK] Modelo exportado exitosamente en: {ARCHIVO_MODELO}")